In [17]:
!pip install -U crewai

In [18]:
!pip install "crewai[google-genai]"

In [19]:
from crewai import Agent,LLM
from google.colab import userdata

gemini_api_key = userdata.get('GEMINI_API_KEY')


llm = LLM(
   model="gemini/gemini-3.5-flash-lite",
   api_key=gemini_api_key
)

game_designer = Agent(
    role="Creative Game Designer",
    goal="Come up with fun, feasible game concepts and detailed mechanics based on user idea",
    backstory=
      "You are an experienced game designer."
      "You excel at turning vague ideas into clear, exciting game designs including:"
      "- core loop, rules, win/lose conditions"
      "- basic entities (player, enemies, items)"
      "- controls and feel"
      "Keep it simple enough to implement in pure Python + Pygame in one file.",
    verbose=True,
    llm=llm,
)

In [20]:
!pip install -U 'crewai[tools]'

In [21]:
from crewai_tools import SerperDevTool

serper_api_key = userdata.get('SERPER_API_KEY')

search_tool = SerperDevTool(api_key=serper_api_key)


senior_engineer = Agent(
   role="Senior Python Game Developer",
   goal="Write clean, working Python code (using Pygame) for the described game",
    backstory=
        "You are a senior software engineer specialized in Python game development with Pygame."
        "You write structured, readable code with:"
        "- Proper game loop, event handling, drawing"
        "- Comments explaining key parts"
        "- Error handling where needed"
        "You always produce a complete, runnable .py file.",
   verbose=True,
   llm=llm,
)


In [22]:
qa_engineer = Agent(
    role="QA Engineer & Code Reviewer",
    goal="Test, review, and improve the code for bugs, playability, and completeness",
    backstory=
        "You are a meticulous QA engineer and code reviewer."
        "You carefully check:"
        "- Does the code run without errors?"
        "- Does it implement ALL the designed features?"
        "- Is it fun/playable? Any obvious balance issues?"
        "- Code style, variable names, comments"
        "Suggest fixes or small improvements and output the FINAL improved code.",
    verbose=True,
    llm=llm,
)


In [23]:
from crewai import Task

In [24]:
task_design = Task(
    description=
        "Take the user's game idea: {game_idea}"
        "1. Clarify and expand it into a fun, simple 2D game"
        "2. Describe: objective, controls, entities, win/lose"
        "3. Keep scope small (one level, basic mechanics)"
        "Output format:"
        "## Game Design Document"
        "- Title: ..."
        "- Genre: ..."
        "- Objective: ..."
        "- Controls: ..."
        "- Entities: ..."
        "- Mechanics: ...",
    expected_output="A clear markdown Game Design Document",
    agent=game_designer
)

In [25]:
task_code = Task(
    description=
        "Using the game design from the previous task"
        "Write a COMPLETE, standalone Python script using Pygame that implements the game."
        "- Include import pygame, sys, random (if needed)"
        "- Full game loop, init, events, update, draw"
        "- Make it runnable with python game.py"
        "- Add simple comments"
        "- The main game loop must be exposed in the python code, it should not be inside any function like main"
        "- Final answer MUST be ONLY the Python code and Instructions on how to play the game",
    expected_output="A complete runnable Pygame Python script",
    agent=senior_engineer,
    context=[task_design]
)

In [26]:
task_review = Task(
    description=
        "Review the Python code from the previous task."
        "1. Check for syntax/runtime errors"
        "2. Verify it matches the design document"
        "3. Test mentally: does it have init, loop, quit handling, drawing?"
        "4. Suggest fixes/improvements if needed"
        "5. Output the FINAL, improved, ready-to-run code"
        "Your final answer MUST be ONLY the complete Python code along with the instructions on how to play the game",
    expected_output="Final polished, runnable Pygame Python script and instructions on how to play the game",
    agent=qa_engineer,
    context=[task_design, task_code]
)

In [27]:

from crewai import Crew, Process

game_crew = Crew(
      agents=[game_designer, senior_engineer, qa_engineer],
      tasks=[task_design, task_code, task_review],
      process=Process.sequential,
      verbose=True
)


In [28]:
game_idea = "A fun endless runner where a character jumps over obstacles"

result = await game_crew.kickoff_async(
    inputs={"game_idea": game_idea}
)

print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0afd5729-4710-40a2-8c40-44dc72a565f8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Take the user's game idea: A fun endless runner where a character jumps over obstacles1. Clarify and     │
│  expand it into a fun, simple 2D game2. Describe: objective, controls, entities, win/lose3. Keep scope small    │
│  (one level, basic mechanics)Output format:## Game Design Document- Title: ...- Genre: ...- Objective: ...-     │
│  Controls: ...- Entities: ...- Mechanics: ...                                                                   │
│  ID: a31d52c4-0a27-4af9-bfbf-2eafe18a7934                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Creative Game Designer                                                                                  │
│                                                                                                                 │
│  Task: Take the user's game idea: A fun endless runner where a character jumps over obstacles1. Clarify and     │
│  expand it into a fun, simple 2D game2. Describe: objective, controls, entities, win/lose3. Keep scope small    │
│  (one level, basic mechanics)Output format:## Game Design Document- Title: ...- Genre: ...- Objective: ...-     │
│  Controls: ...- Entities: ...- Mechanics: ...                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Creative Game Designer                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Game Design Document                                                                                        │
│                                                                                                                 │
│  - **Title:** Neon Dash: Cyber Jumper                                                                           │
│  - **Genre:** 2D Endless Runner                                                                                 │
│  - **Objective:** Survive as long as possible by jumping over incoming neon obstacles. Score points based on    │
│  the distance survived and obstacles successfully cleared. The game speeds up over time to increase the         │
│  challenge.                                                                                                     │
│  - **Controls:**                                                                                                │
│    - `SPACEBAR` or `UP ARROW`: Jump                                                                             │
│    - `DOWN ARROW`: Fast Fall / Duck (optional for advanced obstacles)                                           │
│    - `ESC`: Pause / Quit                                                                                        │
│                                                                                                                 │
│  - **Entities:**                                                                                                │
│    - **Player:** A glowing cyberpunk runner confined to the left side of the screen. Can perform single or      │
│  double jumps. Has a tight rectangular collision box.                                                           │
│    - **Obstacles:** Procedurally spawned cyber-spikes and floating drones moving from right to left at varying  │
│  speeds.                                                                                                        │
│    - **Ground:** A scrolling neon grid floor that gives the player a surface to run on.                         │
│    - **Background:** A parallax-scrolling city skyline with neon aesthetics to create a sense of speed.         │
│    - **UI / Scoreboard:** Displays the current score and high score in real-time at the top of the screen.      │
│                                                                                                                 │
│  - **Mechanics:**                                                                                               │
│    - **Endless Scrolling:** The world moves continuously to the left, creating infinite forward momentum.       │
│    - **Gravity & Jumping:** Pressing jump applies an upward velocity. Holding jump slightly extends air time    │
│  (variable jump height), and a double-jump is allowed once per airtime.                                         │
│    - **Difficulty Scaling:** Game speed and obstacle spawn rates gradually increase every 10 seconds.           │
│    - **Collision & Game Over:** If the player's bounding box intersects with any obstacle's bounding box, it    │
│  triggers an immediate Game Over state, bringing up a "Press R to Restart" screen.                              │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: QA Engineer & Code Reviewer                                                                             │
│                                                                                                                 │
│  Task: Review the Python code from the previous task.1. Check for syntax/runtime errors2. Verify it matches     │
│  the design document3. Test mentally: does it have init, loop, quit handling, drawing?4. Suggest                │
│  fixes/improvements if needed5. Output the FINAL, improved, ready-to-run codeYour final answer MUST be ONLY     │
│  the complete Python code along with the instructions on how to play the game                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: QA Engineer & Code Reviewer                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```python                                                                                                      │
│  import pygame                                                                                                  │
│  import sys                                                                                                     │
│  import random                                                                                                  │
│                                                                                                                 │
│  # Initialize Pygame                                                                                            │
│  pygame.init()                                                                                                  │
│                                                                                                                 │
│  # Constants                                                                                                    │
│  SCREEN_WIDTH = 800                                                                                             │
│  SCREEN_HEIGHT = 450                                                                                            │
│  FPS = 60                                                                                                       │
│                                                                                                                 │
│  # Colors (Cyberpunk Neon Palette)                                                                              │
│  BG_COLOR = (10, 10, 25)                                                                                        │
│  GRID_COLOR = (0, 255, 204)                                                                                     │
│  PLAYER_COLOR = (0, 255, 255)                                                                                   │
│  PLAYER_GLOW = (0, 150, 150)                                                                                    │
│  OBSTACLE_COLOR = (255, 0, 127)                                                                                 │
│  OBSTACLE_GLOW = (150, 0, 75)                                                                                   │
│  TEXT_COLOR = (255, 255, 255)                                                                                   │
│  UI_ACCENT = (255, 230, 0)                                                                                      │
│  BUILDING_COLOR = (20, 20, 45)                                                                                  │
│                                                                                                                 │
│  # Setup Screen                                                                                                 │
│  screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))                                                │
│  pygame.display.set_caption("Neon Dash: Cyber Jumper")                                                          │
│  clock = pygame.time.Clock()                                                                                    │
│                                                                                                                 │
│  # Fonts                                               

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the Python code from the previous task.1. Check for syntax/runtime errors2. Verify it matches     │
│  the design document3. Test mentally: does it have init, loop, quit handling, drawing?4. Suggest                │
│  fixes/improvements if needed5. Output the FINAL, improved, ready-to-run codeYour final answer MUST be ONLY     │
│  the complete Python code along with the instructions on how to play the game                                   │
│  Agent: QA Engineer & Code Reviewer                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

```python
import pygame
import sys
import random

# Initialize Pygame
pygame.init()

# Constants
SCREEN_WIDTH = 800
SCREEN_HEIGHT = 450
FPS = 60

# Colors (Cyberpunk Neon Palette)
BG_COLOR = (10, 10, 25)
GRID_COLOR = (0, 255, 204)
PLAYER_COLOR = (0, 255, 255)
PLAYER_GLOW = (0, 150, 150)
OBSTACLE_COLOR = (255, 0, 127)
OBSTACLE_GLOW = (150, 0, 75)
TEXT_COLOR = (255, 255, 255)
UI_ACCENT = (255, 230, 0)
BUILDING_COLOR = (20, 20, 45)

# Setup Screen
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Neon Dash: Cyber Jumper")
clock = pygame.time.Clock()

# Fonts
font_small = pygame.font.SysFont("Arial", 20)
font_large = pygame.font.SysFont("Arial", 40, bold=True)

# Game State Variables
score = 0
high_score = 0
game_speed = 6.0
speed_timer = 0
game_state = "START" # START, PLAYING, GAMEOVER

# Ground Level
GROUND_Y = 350

# Player Class
class Player:
    def __init__(self):
        self.width = 30
        self.height = 50
        self.x = 100
        

In [29]:
%%writefile main.py

import pygame
import sys
import random

pygame.init()

# Screen settings
WIDTH = 800
HEIGHT = 400
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Endless Runner")

clock = pygame.time.Clock()
font = pygame.font.SysFont(None, 36)

# Colors
SKY = (135, 206, 235)
GROUND = (80, 180, 80)
PLAYER_COLOR = (50, 50, 220)
OBSTACLE_COLOR = (220, 50, 50)
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)

# Ground
GROUND_Y = 330

# Player
player = pygame.Rect(100, GROUND_Y - 50, 40, 50)
player_velocity_y = 0
gravity = 1
jump_strength = -17
on_ground = True

# Obstacles
obstacles = []
obstacle_timer = 0
obstacle_interval = 90

# Game state
score = 0
game_over = False


def reset_game():
    global player, player_velocity_y, on_ground
    global obstacles, obstacle_timer, score, game_over

    player = pygame.Rect(100, GROUND_Y - 50, 40, 50)
    player_velocity_y = 0
    on_ground = True
    obstacles = []
    obstacle_timer = 0
    score = 0
    game_over = False


running = True

while running:
    clock.tick(60)

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_SPACE and on_ground and not game_over:
                player_velocity_y = jump_strength
                on_ground = False

            if event.key == pygame.K_r and game_over:
                reset_game()

    if not game_over:
        # Player movement
        player_velocity_y += gravity
        player.y += player_velocity_y

        if player.bottom >= GROUND_Y:
            player.bottom = GROUND_Y
            player_velocity_y = 0
            on_ground = True

        # Create obstacles
        obstacle_timer += 1

        if obstacle_timer >= obstacle_interval:
            obstacle_width = random.randint(25, 45)
            obstacle_height = random.randint(35, 65)

            obstacle = pygame.Rect(
                WIDTH,
                GROUND_Y - obstacle_height,
                obstacle_width,
                obstacle_height
            )

            obstacles.append(obstacle)
            obstacle_timer = 0
            obstacle_interval = random.randint(70, 120)

        # Move obstacles
        for obstacle in obstacles:
            obstacle.x -= 6

        # Remove obstacles outside the screen
        obstacles = [
            obstacle for obstacle in obstacles
            if obstacle.right > 0
        ]

        # Collision detection
        for obstacle in obstacles:
            if player.colliderect(obstacle):
                game_over = True

        score += 1

    # Draw background
    screen.fill(SKY)

    # Draw ground
    pygame.draw.rect(
        screen,
        GROUND,
        (0, GROUND_Y, WIDTH, HEIGHT - GROUND_Y)
    )

    # Draw player
    pygame.draw.rect(screen, PLAYER_COLOR, player)

    # Draw obstacles
    for obstacle in obstacles:
        pygame.draw.rect(screen, OBSTACLE_COLOR, obstacle)

    # Draw score
    score_text = font.render(
        f"Score: {score // 10}",
        True,
        BLACK
    )
    screen.blit(score_text, (20, 20))

    # Game-over message
    if game_over:
        game_over_text = font.render(
            "Game Over! Press R to Restart",
            True,
            BLACK
        )
        text_rect = game_over_text.get_rect(
            center=(WIDTH // 2, HEIGHT // 2)
        )
        screen.blit(game_over_text, text_rect)

    pygame.display.flip()

pygame.quit()
sys.exit()

Overwriting main.py


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0afd5729-4710-40a2-8c40-44dc72a565f8                                                                       │
│  Final Output: ```python                                                                                        │
│  import pygame                                                                                                  │
│  import sys                                                                                                     │
│  import random                                                                                                  │
│                                                                                                                 │
│  # Initialize Pygame                                                                                            │
│  pygame.init()                                                                                                  │
│                                                                                                                 │
│  # Constants                                                                                                    │
│  SCREEN_WIDTH = 800                                                                                             │
│  SCREEN_HEIGHT = 450                                                                                            │
│  FPS = 60                                                                                                       │
│                                                                                                                 │
│  # Colors (Cyberpunk Neon Palette)                                                                              │
│  BG_COLOR = (10, 10, 25)                                                                                        │
│  GRID_COLOR = (0, 255, 204)                                                                                     │
│  PLAYER_COLOR = (0, 255, 255)                                                                                   │
│  PLAYER_GLOW = (0, 150, 150)                                                                                    │
│  OBSTACLE_COLOR = (255, 0, 127)                                                                                 │
│  OBSTACLE_GLOW = (150, 0, 75)                                                                                   │
│  TEXT_COLOR = (255, 255, 255)                                                                                   │
│  UI_ACCENT = (255, 230, 0)                                                                                      │
│  BUILDING_COLOR = (20, 20, 45)                                                                                  │
│                                                                                                                 │
│  # Setup Screen                                                                                                 │
│  screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))                                                │
│  pygame.display.set_caption("Neon Dash: Cyber Jumper")                                                          │
│  clock = pygame.time.Clock()                                                                                    │
│                                                                                                                 │
│  # Fonts                                              

# **Do not edit any of the below code**

In [30]:
# Install required packages
!pip install pygbag pyngrok -q

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# **How to get pyngrok API Key?**
- Go to https://dashboard.ngrok.com/get-started/your-authtoken
- Copy your authtoken
- Run this cell with YOUR token:

In [31]:
from pyngrok import ngrok
import os
import signal

ngrok.kill()

# Stop any existing HTTP server using port 8000
os.system("fuser -k 8000/tcp")

print("Existing ngrok tunnel and server stopped.")

Existing ngrok tunnel and server stopped.


In [32]:
import subprocess
import time
from pyngrok import ngrok

# Step 1: Build the game
print("🔨 Building game...")
build_result = subprocess.run(
    ['pygbag', '--build', '--version', '0.9', '--PYBUILD', '3.12', '--cdn', 'https://pygame-web.github.io/archives/0.9/', 'main.py'],
    capture_output=True, text=True
)
print(build_result.stdout)

if build_result.returncode != 0:
    print("❌ Build failed:")
    print(build_result.stderr)
else:
    print("✅ Build successful!")

    # Step 2: Start HTTP server
    print("\n🚀 Starting server...")
    server = subprocess.Popen(
        ['python', '-m', 'http.server', '8000', '--directory', '/content/build/web'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    # Wait for server to start
    time.sleep(5)

    # Step 3: Create ngrok tunnel
    try:
        print("🌐 Creating public URL...")
        public_url = ngrok.connect(8000)

        print("\n" + "="*60)
        print("🎮 YOUR GAME IS READY!")
        print("="*60)
        print(f"\n🔗 Click here to play: {public_url}")
        print("\n📝 How to play:")
        print("   • Press SPACE or Click to jump")
        print("="*60)

    except Exception as e:
        print(f"\n❌ Error creating tunnel: {e}")
        print("\n💡 Make sure you ran Cell 3 with a valid ngrok token")

🔨 Building game...

Serving python files from [/content/build/web]

with no security/performance in mind, i'm just a test tool : don't rely on me


SUMMARY
________________________

# the app folder
app_folder=/content

# artefacts directory
build_dir=/content/build/web

# cache directory
cache=/content/build/web-cache

# the window title and icon name
app_name=content

# package name, better make it unique
package=web.pygame.content-1789298957

# icons:  96x96 for desktop, 16x16 for web
icon=favicon.png

# js/wasm provider
cdn=https://pygame-web.github.io/archives/0.9/

now packing application ....


Ignored dirs: []
Ignored files: []
Now in /
     /main.py
Now in /sample_data
     /sample_data/README.md
     /sample_data/anscombe.json
     /sample_data/mnist_test.csv
     /sample_data/california_housing_train.csv
     /sample_data/mnist_train_small.csv
     /sample_data/california_housing_test.csv
optimizing /content
	/content : main.py
	/content : sample_data/README.md
	/content : s